In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import json
import time

import numpy as np
import networkx as nx
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import fleet_opt_core

import src
import figures

from src.rust_bridge import build_rust_network
from src.reload import deep_reload
from figures import style
matplotlib.rcParams.update(**style['rcparams'])

## Constrained optimization example

In [ ]:
'''
Loading graphs

trips - a graph with trips as nodes and edges to possible successors
locations - a graph representing the physical cost of traveling between
    trip terminals and depots
'''
deep_reload(src)

trips = src.graph.graph_from_json('Data/Unitrans/trips.json')
locations = src.graph.graph_from_json('Data/Unitrans/locations.json')

In [ ]:
'''
Loading vehicles and ports
'''

with open('Data/vehicle_types_reduced.json', 'r') as f:
    
    vehicles_data = json.load(f)

vehicle_types = {
    v['type']: src.optimization.Vehicle_Type(**v) for v in vehicles_data
}

with open('Data/port_types_reduced.json', 'r') as f:
    
    ports_data = json.load(f)

port_types = {
    v['type']: src.optimization.Port_Type(**v) for v in ports_data
}

In [ ]:
'''
Making a Python Network

In this example the objectives are initial and daily cost and the constraints
are energy (applied on the tour level) lot size (limiting the number of buses
for each depot), and a requirement that 25% of the fleet be non-ICEVs.
'''
deep_reload(src)

objectives = {
    'Initial Cost': src.optimization.Initial_Cost(),
    'Daily Cost': src.optimization.Daily_Cost(),
}

constraints = {
    'Energy': src.optimization.Energy(),
    'Lot Size': src.optimization.Lot_Size_Limit(
        sizes = {
            'Primary Depot': 50,
            'Secondary Depot': 50,
        },
    ),
    'Portion': src.optimization.Fleet_Portion(
    	included = ['SRBEV', 'MRBEV', 'LRBEV', 'FCEV'],
    	portion = .25,
	),
}


kw = {
    'trips': trips,
    'locations': locations,
    'vehicle_types': vehicle_types,
    'port_types': port_types,
    'constraints': constraints,
    'objectives': objectives,
}

network = src.optimization.Network(**kw)

In [ ]:
'''
Running the optimization in Python (Not Reccommended) - This will take
    around 30 minutes
'''

t0 = time.perf_counter()

kw = {
    'seed': 460780573,
    'max_iter': 1000,
    'min_iter': 500,
    'initial_population_size': 1500,
    'population_size': 250,
    'crossover_probability': 0.5,
    'mutation_probability': 0.05,
    'survival_threshold': 0,
    'retries': 5,
    'store_interval': 1,
    'same_depot': True,
}

population, generations = network.optimize(**kw)
generations = {i: generations[i] for i in np.sort(list(generations.keys()))}

elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s  |  {len(generations)} generations stored')

In [ ]:
'''
Making the Rust Network from the Python network
'''
deep_reload(src)

rust_net, index_maps = src.rust_bridge.build_rust_network(network)
print(rust_net.status())

In [ ]:
'''
Running the optimization in Rust - This will take about one minute
'''

t0 = time.perf_counter()

kw = {
    'seed': 460780573,
    'max_iter': 1000,
    'min_iter': 500,
    'initial_population_size': 1500,
    'population_size': 250,
    'crossover_probability': 0.5,
    'mutation_probability': 0.05,
    'survival_threshold': 0,
    'retries': 5,
    'store_interval': 1,
    'same_depot': True,
}

population, generations = rust_net.optimize(**kw)
generations = {i: generations[i] for i in np.sort(list(generations.keys()))}

elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s  |  {len(generations)} generations stored')

In [ ]:
'''
Plotting generations
'''
deep_reload(src)
deep_reload(figures)

fig, ax = plt.subplots(figsize = (8, 5))

show = [10, 100, 300, len(generations) - 1]

cmap = src.plot.Colormap(
    colors = 'ibm', vmin = 0, vmax = len(show) - 1
)

ax = figures.generations_plot(
    ax, generations, 'Initial Cost', 'Daily Cost',
    interval = 200, rank = 100, cmap = cmap, initial = True,
    show = show,
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
}

_ = ax.set(**kw)

kw = {
    'ls': '--',
}

_ = ax.grid(**kw)

In [ ]:
deep_reload(src)

rs_pop_dict = src.rust_bridge.rust_pop_to_python_format(population, index_maps)

py_solutions = src.rust_bridge.resolve_pareto_front(
    network, rs_pop_dict, index_maps, max_solutions = 250, rank = 0,
)
py_pop_dict  = src.optimization.population_to_dict(py_solutions)

In [ ]:
'''
Plotting Pareto Compositions
'''
deep_reload(src)
deep_reload(figures)

matplotlib.rcParams.update(**style['rcparams'])
fig, ax = plt.subplots(1, 2, figsize = (10, 4.5))

n_shown = 15

ax[0] = figures.vehicles_scatter_pie_int(
    ax[0], py_pop_dict, n_shown = n_shown, size = .04, rank = 0,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['ICEV', 'FCEV', 'SRBEV', 'MRBEV', 'LRBEV'],
)

ax[1] = figures.ports_scatter_pie_int(
    ax[1], py_pop_dict, n_shown = n_shown, size = .03, rank = 0,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['DIESEL-PUMP', 'LH2-PUMP', 'AC-PLUG', 'DC-PLUG'],
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[0].set(**kw)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[1].set(**kw)

kw = {
    'ls': '--',
}

_ = [ax.grid(**kw) for ax in ax]
_ = [ax.legend() for ax in ax]

## Integrating a buffer of uncertatinty for trip duration

In [ ]:
'''
The previous example assumes all trips end on time. However, if Unitrans can only
guarantee that all trips will end within 15% of their nominal time then certain
edges become infeasible. These edges are removed and the optimization is re-run
'''

deep_reload(src)

trips = src.graph.trip_edge_weights(trips, locations)

trips_pruned = src.graph.prune_by_time_buffer(
    trips.copy(), locations, buffer = .25
)

trips.number_of_edges(), trips_pruned.number_of_edges()

In [ ]:
'''
Making a Python Network

In this example the objectives are initial and daily cost and the constraints
are energy (applied on the tour level) lot size (limiting the number of buses
for each depot), and a requirement that 25% of the fleet be non-ICEVs.
'''
deep_reload(src)

objectives = {
    'Initial Cost': src.optimization.Initial_Cost(),
    'Daily Cost': src.optimization.Daily_Cost(),
}

constraints = {
    'Energy': src.optimization.Energy(),
    'Lot Size': src.optimization.Lot_Size_Limit(
        sizes = {
            'Primary Depot': 50,
            'Secondary Depot': 50,
        },
    ),
    'Portion': src.optimization.Fleet_Portion(
    	included = ['SRBEV', 'MRBEV', 'LRBEV', 'FCEV'],
    	portion = .25,
	),
}


kw = {
    'trips': trips_pruned,
    'locations': locations,
    'vehicle_types': vehicle_types,
    'port_types': port_types,
    'constraints': constraints,
    'objectives': objectives,
}

network = src.optimization.Network(**kw)

In [ ]:
'''
Making the Rust Network from the Python network
'''
deep_reload(src)

rust_net, index_maps = src.rust_bridge.build_rust_network(network)
print(rust_net.status())

In [ ]:
'''
Running the optimization in Rust - This will take about one minute
'''

t0 = time.perf_counter()

kw = {
    'seed': 460780573,
    'max_iter': 1000,
    'min_iter': 500,
    'initial_population_size': 1500,
    'population_size': 250,
    'crossover_probability': 0.5,
    'mutation_probability': 0.05,
    'survival_threshold': 0,
    'retries': 5,
    'store_interval': 1,
    'same_depot': True,
}

population, generations = rust_net.optimize(**kw)
generations = {i: generations[i] for i in np.sort(list(generations.keys()))}

elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s  |  {len(generations)} generations stored')

In [ ]:
'''
Plotting generations
'''
deep_reload(src)
deep_reload(figures)

fig, ax = plt.subplots(figsize = (8, 5))

show = [10, 100, 300, len(generations) - 1]

cmap = src.plot.Colormap(
    colors = 'ibm', vmin = 0, vmax = len(show) - 1
)

ax = figures.generations_plot(
    ax, generations, 'Initial Cost', 'Daily Cost',
    interval = 200, rank = 100, cmap = cmap, initial = True,
    show = show,
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
}

_ = ax.set(**kw)

kw = {
    'ls': '--',
}

_ = ax.grid(**kw)

In [ ]:
deep_reload(src)

rs_pop_dict = src.rust_bridge.rust_pop_to_python_format(population, index_maps)

py_solutions = src.rust_bridge.resolve_pareto_front(
    network, rs_pop_dict, index_maps, max_solutions = 250, rank = 0,
)
py_pop_dict  = src.optimization.population_to_dict(py_solutions)

In [ ]:
'''
Plotting Pareto Compositions
'''
deep_reload(src)
deep_reload(figures)

matplotlib.rcParams.update(**style['rcparams'])
fig, ax = plt.subplots(1, 2, figsize = (10, 4.5))

n_shown = 15

ax[0] = figures.vehicles_scatter_pie_int(
    ax[0], py_pop_dict, n_shown = n_shown, size = .04, rank = 0,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['ICEV', 'FCEV', 'SRBEV', 'MRBEV', 'LRBEV'],
)

ax[1] = figures.ports_scatter_pie_int(
    ax[1], py_pop_dict, n_shown = n_shown, size = .03, rank = 0,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['DIESEL-PUMP', 'LH2-PUMP', 'AC-PLUG', 'DC-PLUG'],
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[0].set(**kw)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[1].set(**kw)

kw = {
    'ls': '--',
}

_ = [ax.grid(**kw) for ax in ax]
_ = [ax.legend() for ax in ax]

## Technically - you can also run single-objective optimization

In [ ]:
'''
Technically this also works for single-objective
'''
deep_reload(src)

objectives = {
    'Initial Cost': src.optimization.Initial_Cost(),
}

constraints = {
    'Energy': src.optimization.Energy(),
    'Lot Size': src.optimization.Lot_Size_Limit(
        sizes = {
            'Primary Depot': 50,
            'Secondary Depot': 50,
        },
    ),
    'Portion': src.optimization.Fleet_Portion(
    	included = ['SRBEV', 'MRBEV', 'LRBEV', 'FCEV'],
    	portion = .25,
	),
}


kw = {
    'trips': trips,
    'locations': locations,
    'vehicle_types': vehicle_types,
    'port_types': port_types,
    'constraints': constraints,
    'objectives': objectives,
}

network = src.optimization.Network(**kw)

In [ ]:
'''
Making the Rust Network from the Python network
'''
deep_reload(src)

rust_net, index_maps = src.rust_bridge.build_rust_network(network)
print(rust_net.status())

In [ ]:
'''
Running the optimization in Rust - This will take about one minute
'''

t0 = time.perf_counter()

kw = {
    'seed': 460780573,
    'max_iter': 1000,
    'min_iter': 500,
    'initial_population_size': 1500,
    'population_size': 250,
    'crossover_probability': 0.5,
    'mutation_probability': 0.05,
    'survival_threshold': 0,
    'retries': 5,
    'store_interval': 1,
    'same_depot': True,
}

population, generations = rust_net.optimize(**kw)
generations = {i: generations[i] for i in np.sort(list(generations.keys()))}

elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s  |  {len(generations)} generations stored')

In [ ]:
'''
Adding generation as a "second objective" for plotting
'''

for i, g in generations.items():
    for k, v in g.items():

        v['fitness']['Daily Cost'] = v['fitness']['Initial Cost'] * 0 + i

In [ ]:
'''
Plotting generations
'''
deep_reload(src)
deep_reload(figures)

fig, ax = plt.subplots(figsize = (8, 5))

show = [10, 100, 300, len(generations) - 1]

cmap = src.plot.Colormap(
    colors = 'ibm', vmin = 0, vmax = len(show) - 1
)

ax = figures.generations_plot(
    ax, generations, 'Initial Cost', 'Daily Cost',
    interval = 200, rank = 100, cmap = cmap, initial = True,
    show = show,
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
}

_ = ax.set(**kw)

kw = {
    'ls': '--',
}

_ = ax.grid(**kw)

In [ ]:
deep_reload(src)

rs_pop_dict = src.rust_bridge.rust_pop_to_python_format(population, index_maps)

py_solutions = src.rust_bridge.resolve_pareto_front(
    network, rs_pop_dict, index_maps, max_solutions = 250, rank = 15,
)
py_pop_dict  = src.optimization.population_to_dict(py_solutions)

In [ ]:
'''
Adding generation as a "second objective" for plotting
'''

for k, v in py_pop_dict.items():

    v['fitness']['Daily Cost'] = 1 / (int(k) + 1)

In [ ]:
'''
Plotting Pareto Compositions
'''
deep_reload(src)
deep_reload(figures)

matplotlib.rcParams.update(**style['rcparams'])
fig, ax = plt.subplots(1, 2, figsize = (10, 4.5))

n_shown = 15

ax[0] = figures.vehicles_scatter_pie_int(
    ax[0], py_pop_dict, n_shown = n_shown, size = .04, rank = 100,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['ICEV', 'FCEV', 'SRBEV', 'MRBEV', 'LRBEV'],
)

ax[1] = figures.ports_scatter_pie_int(
    ax[1], py_pop_dict, n_shown = n_shown, size = .03, rank = 100,
    cmap = src.plot.Colormap('ibm'), exponent = 1,
    keys = ['DIESEL-PUMP', 'LH2-PUMP', 'AC-PLUG', 'DC-PLUG'],
)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    'ylabel': 'Daily Cost [Thousand USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[0].set(**kw)

kw = {
    'xlabel': 'Capital Cost [Million USD]',
    # 'xticks': [30, 40, 50, 60, 70, 80, 90],
}

_ = ax[1].set(**kw)

kw = {
    'ls': '--',
}

_ = [ax.grid(**kw) for ax in ax]
_ = [ax.legend() for ax in ax]